# Notebook 05: GradCAM++ Implementation and Evaluation

GradCAM++ visualisation across all six inpainted-arm models (three architectures × two training
modes). Produces heatmap overlays for a stratified image subset, computes quantitative
localisation scores (IoU) against radiologist-annotated segmentation masks, and exports
materials for radiologist qualitative evaluation.

Answers RQ3: Do gradient-based explainability methods produce visualisations that align with
radiologists' clinical intuition and knowledge?


## 1. Setup

Install dependencies, import libraries, define path constants and runtime flags.

`FORCE_REGENERATE`: when `False`, expensive generation steps are skipped if their output files already exist on disk. Set to `True` to rerun from scratch.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install opencv-python -q
!pip install grad-cam -q

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import torch
import torch.nn.functional as F
from torchvision import transforms

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
# Root & existing directories
ROOT    = Path('/content/drive/MyDrive/University/Masters/MSc_Thesis')
RESULTS = ROOT / 'outputs' / 'results'
SRC     = ROOT / 'src'
RAW     = ROOT / 'data' / 'raw'
FINAL   = ROOT / 'data' / 'final'

# GradCAM output directory
GRADCAM_DIR = ROOT / 'outputs' / 'gradcam'
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

# src/ on path
sys.path.append(str(SRC))

from models        import build_model, NUM_CLASSES
from utils         import SEED, seed_everything
from dataset_utils import (CLASS_TO_IDX, IMAGENET_MEAN, IMAGENET_STD,
                            build_transform, scan_class_folders)

seed_everything(SEED)

# Runtime flag
FORCE_REGENERATE = False
RADIOLOGIST_PACKET_LOCKED = True

# Path verification
print('Path verification:')
for name, path in [
    ('ROOT',        ROOT),
    ('RESULTS',     RESULTS),
    ('SRC',         SRC),
    ('RAW',         RAW),
    ('FINAL',       FINAL),
    ('GRADCAM_DIR', GRADCAM_DIR),
]:
    print(f'  {name}: {"Found" if path.exists() else "NOT FOUND"}')

## 2. Data Staging & Mask Loading

Builds reusable, verified functions to locate, load, and spatially align radiologist-annotated segmentation masks with their corresponding preprocessed images. No bulk mask processing happens in this section, only construction and verification of the pipeline. Masks are loaded on demand in later sections, for whichever images the Section 4 stratified selection actually includes.

The central correctness requirement: a mask must land on the exact same 224×224 canvas position as its paired preprocessed image. This is achieved by re-deriving the identical scale and padding geometry N02 used (`to_final_format`), applied to the mask's own native dimensions, with nearest-
neighbour interpolation to preserve hard binary edges.

### 2.1 Registries

Loads the master image registry (`df_clean_post_exclusion.csv`) and the inference predictions (`predictions.csv`), filtered to the inpainted arm only. Deduplicated to unique `(dataset, image_name)` pairs, since masks do not depend on which of the six models made the prediction.

In [ ]:
# Load registries
df_clean = pd.read_csv(RESULTS / 'df_clean_post_exclusion.csv')

EVAL_DIR = RESULTS / 'eval'
predictions_df = pd.read_csv(EVAL_DIR / 'predictions.csv')

# eval_data -> raw dataset name mapping
EVAL_DATA_TO_DATASET = {
    'busi_test': 'BUSI',
    'bus_uclm':  'BUS-UCLM',
    'breast':    'BrEaST',
}

# Inverse CLASS_TO_IDX: numeric true_label -> lowercase class string
IDX_TO_CLASS = {v: k.lower() for k, v in CLASS_TO_IDX.items()}

# Filter to inpainted arm, dedupe to unique images
inpainted_preds = predictions_df[predictions_df['arm'] == 'inpainted'].copy()

unique_images = (
    inpainted_preds[['eval_data', 'image_name', 'true_label']]
    .drop_duplicates()
    .reset_index(drop=True)
)
unique_images['dataset']    = unique_images['eval_data'].map(EVAL_DATA_TO_DATASET)
unique_images['class_name'] = unique_images['true_label'].map(IDX_TO_CLASS)

assert unique_images['dataset'].isna().sum() == 0, 'Unmapped eval_data value found.'
assert unique_images['class_name'].isna().sum() == 0, 'Unmapped true_label value found.'

print(f'inpainted_preds: {len(inpainted_preds)} rows (6 models × unique images)')
print(f'unique_images:   {len(unique_images)} rows (deduplicated)')
print(unique_images.groupby(['dataset', 'class_name'])['image_name'].count().to_string())

### 2.2 Raw & Final Path Definitions

In [ ]:
# Raw paths
BUSI_RAW      = RAW / 'BUSI' / 'Dataset_BUSI' / 'Dataset_BUSI_with_GT'
BUSUCLM_RAW   = RAW / 'BUS-UCLM' / 'BUS-UCLM Breast ultrasound lesion segmentation dataset' / 'BUS-UCLM'
BUSUCLM_IMAGES_RAW = BUSUCLM_RAW / 'images'
BUSUCLM_MASKS_RAW  = BUSUCLM_RAW / 'masks'
BREAST_RAW    = RAW / 'BrEaST' / 'BrEaST-Lesions' / 'BrEaST-Lesions_USG-images_and_masks'

#  Final (preprocessed, 224x224) paths
BUSI_FINAL    = FINAL / 'BUSI' / 'BUSI_inpainted'
BUSUCLM_FINAL = FINAL / 'BUS-UCLM'
BREAST_FINAL  = FINAL / 'BrEaST'

print('Path verification:')
for name, path in [
    ('BUSI raw',        BUSI_RAW),
    ('BUS-UCLM images',BUSUCLM_IMAGES_RAW),
    ('BUS-UCLM masks', BUSUCLM_MASKS_RAW),
    ('BrEaST raw',       BREAST_RAW),
    ('BUSI final',       BUSI_FINAL),
    ('BUS-UCLM final',   BUSUCLM_FINAL),
    ('BrEaST final',     BREAST_FINAL),
]:
    print(f'  {name}: {"Found" if path.exists() else "NOT FOUND"}')

### 2.3 Mask Locators

One function per dataset, since naming and folder structure differ completely across the three.
Each returns a **list** of matching raw mask file paths (empty if none found — e.g. normal-class images, which have no lesion).

- BUSI: `<class>/` subfolder; `{stem}_mask.png`, plus `{stem}_mask_1.png` etc. for multi-lesion images.
- BUS-UCLM: flat `masks/` folder; filename identical to the image.
- BrEaST: flat folder; `{stem}_tumor.png` for the primary lesion. Secondary lesions (`{stem}_other1.png` etc.) are located separately and are **not** part of the primary mask.

In [ ]:
def locate_busi_masks(image_name, class_name):
    stem = Path(image_name).stem
    folder = BUSI_RAW / class_name
    return sorted(folder.glob(f'{stem}_mask*.png'))


def locate_busuclm_mask(image_name):
    path = BUSUCLM_MASKS_RAW / image_name
    return [path] if path.exists() else []


def locate_breast_masks(image_name):
    stem = Path(image_name).stem
    primary_path = BREAST_RAW / f'{stem}_tumor.png'
    primary = [primary_path] if primary_path.exists() else []
    secondary = sorted(BREAST_RAW.glob(f'{stem}_other*.png'))
    return primary, secondary


def locate_masks(dataset, image_name, class_name):
    """
    Unified dispatcher. Returns (primary_mask_paths, secondary_mask_paths).
    secondary_mask_paths is always [] except for BrEaST has_other_lesion cases.
    """
    if dataset == 'BUSI':
        return locate_busi_masks(image_name, class_name), []
    elif dataset == 'BUS-UCLM':
        return locate_busuclm_mask(image_name), []
    elif dataset == 'BrEaST':
        return locate_breast_masks(image_name)
    else:
        raise ValueError(f'Unknown dataset: {dataset}')

### 2.4 Mask Loading, Channel Extraction & Letterbox Alignment

`letterbox_mask` deliberately mirrors N02's `to_final_format` geometry exactly, same scale formula, same centred padding, but with `INTER_NEAREST` instead of `INTER_AREA`/`INTER_LINEAR`, so binary lesion boundaries are not blurred into fractional grey values. Given identical source dimensions, this guarantees the mask lands on the same canvas position as its paired image.

In [ ]:
def load_mask_file(mask_path, threshold=127):
    """
    Loads a mask file as single-channel grayscale and binarises it.
    Safe for BUSI/BrEaST (already near-binary) regardless of stored mode (L or RGB-with-equal-channels).
    """
    arr = np.array(Image.open(mask_path).convert('L'))
    return (arr > threshold).astype(np.uint8)


def load_busuclm_mask_channel(mask_path, class_name, threshold=127):
    """
    BUS-UCLM masks are RGB-coded: green = benign, red = malignant. Extracts the
    single relevant channel based on the image's known class, then binarises.
    """
    arr = np.array(Image.open(mask_path).convert('RGB'))
    if class_name == 'benign':
        channel = arr[:, :, 1]  # green
    elif class_name == 'malignant':
        channel = arr[:, :, 0]  # red
    else:
        raise ValueError(f'No mask channel defined for class: {class_name}')
    return (channel > threshold).astype(np.uint8)


def union_masks(mask_list):
    """Logical OR across multiple binary masks of identical shape (BUSI multi-lesion case)."""
    shapes = {m.shape for m in mask_list}
    assert len(shapes) == 1, f'Cannot union masks of differing shapes: {shapes}'
    return np.logical_or.reduce(mask_list).astype(np.uint8)


def letterbox_mask(mask_bin, target_size=224):
    """
    Replicates to_final_format's exact scale/padding geometry, INTER_NEAREST instead
    of INTER_AREA/INTER_LINEAR. Given the same source (h, w), output aligns pixel-for-pixel
    with the corresponding preprocessed image's canvas.
    """
    h, w = mask_bin.shape
    scale = target_size / max(h, w)
    new_h, new_w = max(int(round(h * scale)), 1), max(int(round(w * scale)), 1)

    resized = cv2.resize(mask_bin.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST)

    canvas = np.zeros((target_size, target_size), dtype=np.uint8)
    top = (target_size - new_h) // 2
    left = (target_size - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = resized

    assert set(np.unique(canvas)).issubset({0, 1}), 'Letterboxed mask is not binary.'
    return canvas


def get_final_image_path(dataset, image_name, class_name):
    cls = class_name.lower()
    if dataset == 'BUSI':
        return BUSI_FINAL / cls / image_name
    elif dataset == 'BUS-UCLM':
        return BUSUCLM_FINAL / cls / image_name
    elif dataset == 'BrEaST':
        return BREAST_FINAL / cls / image_name
    else:
        raise ValueError(f'Unknown dataset: {dataset}')


def load_final_image(dataset, image_name, class_name):
    path = get_final_image_path(dataset, image_name, class_name)
    assert path.exists(), f'Final image not found: {path}'
    return np.array(Image.open(path).convert('RGB'))

### 2.5 Unified Mask Retrieval

Full pipeline for a single image: locate raw mask file(s) → load/union/channel-extract → live shape-check against `df_clean`'s recorded dimensions → letterbox-align to 224×224.
Returns `found=False` (not an error) when no mask exists — the expected case for normal-class images, which are already excluded from quantitative analysis.

In [ ]:
# Fast (dataset, image_name) -> (height, width) lookup
assert df_clean.duplicated(subset=['dataset', 'image_name']).sum() == 0, \
    'Duplicate (dataset, image_name) rows in df_clean -- investigate before proceeding.'
dims_lookup = df_clean.set_index(['dataset', 'image_name'])[['height', 'width']]


def get_lesion_mask(dataset, image_name, class_name, target_size=224):
    """
    Returns a dict:
      found            : bool
      mask             : np.ndarray[224,224] uint8 (0/1), primary lesion mask, or None
      secondary_mask   : np.ndarray[224,224] uint8 (0/1), BrEaST 'other lesion' mask, or None
      warning          : str or None
    """
    result = {'found': False, 'mask': None, 'secondary_mask': None, 'warning': None}

    # Normal-class images have no lesion, by definition, regardless of whether a raw mask
    # file happens to exist on disk. BUS-UCLM and (likely) BUSI store an all-black mask file
    # even for normal images; BrEaST omits the file entirely. Short-circuiting here, uniformly
    # across all three datasets, avoids both the BUS-UCLM channel-extraction crash (there is no
    # lesion class to pick a colour channel for) and a silent BUSI failure mode where an all-zero
    # "lesion" mask would otherwise load without error and corrupt EBPG with meaningless zeros.
    if class_name == 'normal':
        result['warning'] = f'{dataset}/{image_name}: normal class, no lesion mask by definition.'
        return result

    primary_paths, secondary_paths = locate_masks(dataset, image_name, class_name)

    if not primary_paths:
        result['warning'] = f'No mask file found for {dataset}/{image_name}.'
        return result

    expected_h, expected_w = dims_lookup.loc[(dataset, image_name)]

    if dataset == 'BUS-UCLM':
        raw_masks = [load_busuclm_mask_channel(p, class_name) for p in primary_paths]
    else:
        raw_masks = [load_mask_file(p) for p in primary_paths]

    for m in raw_masks:
        assert m.shape == (expected_h, expected_w), (
            f'Mask shape {m.shape} does not match df_clean-recorded '
            f'({expected_h}, {expected_w}) for {dataset}/{image_name}.'
        )

    combined = union_masks(raw_masks) if len(raw_masks) > 1 else raw_masks[0]
    result['mask'] = letterbox_mask(combined, target_size)
    result['found'] = True

    if secondary_paths:
        sec_raw = [load_mask_file(p) for p in secondary_paths]
        for m in sec_raw:
            assert m.shape == (expected_h, expected_w), \
                f'Secondary mask shape mismatch for {dataset}/{image_name}.'
        sec_combined = union_masks(sec_raw) if len(sec_raw) > 1 else sec_raw[0]
        result['secondary_mask'] = letterbox_mask(sec_combined, target_size)

    return result

### 2.6 Sanity Check: Visual Alignment Verification

Kept permanently, re-run every session. Loads one benign and one malignant image per dataset, retrieves the aligned mask, and overlays both on the corresponding preprocessed image. Confirms by eye and by assertion that mask and image occupy the same canvas geometry.

In [ ]:
sample_targets = []
for dataset in ['BUSI', 'BUS-UCLM', 'BrEaST']:
    for cls in ['benign', 'malignant']:
        candidates = unique_images[
            (unique_images['dataset'] == dataset) & (unique_images['class_name'] == cls)
        ]
        assert len(candidates) > 0, f'No candidates found for {dataset}/{cls}.'
        row = candidates.sample(1, random_state=SEED).iloc[0]
        sample_targets.append((dataset, row['image_name'], cls))

fig, axes = plt.subplots(len(sample_targets), 3, figsize=(9, 3 * len(sample_targets)))

for i, (dataset, image_name, cls) in enumerate(sample_targets):
    image = load_final_image(dataset, image_name, cls)
    result = get_lesion_mask(dataset, image_name, cls)

    assert result['found'], f'Sanity check failed: no mask found for {dataset}/{image_name}.'
    mask = result['mask']
    assert mask.shape == image.shape[:2], \
        f'Shape mismatch: image {image.shape[:2]} vs mask {mask.shape} for {dataset}/{image_name}.'

    coverage = mask.mean()
    assert 0.0 < coverage < 0.95, \
        f'Implausible mask coverage ({coverage:.1%}) for {dataset}/{image_name} -- investigate.'

    axes[i, 0].imshow(image); axes[i, 0].set_title(f'{dataset}/{cls}\n{image_name}', fontsize=8)
    axes[i, 1].imshow(mask, cmap='gray'); axes[i, 1].set_title('Aligned mask', fontsize=8)
    axes[i, 2].imshow(image); axes[i, 2].imshow(mask, cmap='Reds', alpha=0.4)
    axes[i, 2].set_title(f'Overlay ({coverage:.1%} coverage)', fontsize=8)

    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
plt.show()

print('Section 2 sanity check passed: all 6 samples aligned, shapes match, coverage plausible.')

## 3. Target Layer Definition & Library Verification

Defines the GradCAM++ target layer for each architecture, sourced directly from the module structure confirmed in `src/models.py`'s own `freeze_*` functions — not guessed from generic torchvision documentation. Verifies each target layer's actual output shape via a forward hook, then runs a real end-to-end GradCAM++ smoke test (real checkpoint, real image, real backward pass) for one model per architecture before Section 4 commits to a full image selection.

Only TL-mode models are used for verification in this section. Target layer structure is identical between TL and scratch for a given architecture — `build_model` calls the same `ARCHITECTURE_BUILDERS[architecture]` function regardless of mode, and freezing only changes `requires_grad` flags, never the module tree. Scratch-mode normalisation statistics differ from TL's ImageNet stats and are handled when scratch models are actually loaded, in Section 5.

### 3.1 Inpainted-Arm Model Registry

Loads `notebook03_results_summary.csv`, filtered to the 6 inpainted-arm rows (3 architectures × 2 modes). This is the same registry N04 used to build `eval_registry`, so checkpoint paths are guaranteed consistent with what produced `predictions.csv`.

In [ ]:
results_summary = pd.read_csv(RESULTS / 'notebook03_results_summary.csv')
final_models = json.load(open(RESULTS / 'final_models.json'))

inpainted_registry = results_summary[results_summary['arm'] == 'inpainted'].reset_index(drop=True)
assert len(inpainted_registry) == 6, f'Expected 6 inpainted-arm rows, got {len(inpainted_registry)}.'

# Cross-check against final_models.json, same drift check N04 performed
for _, row in inpainted_registry.iterrows():
    key = f"{row['architecture']}_{row['mode']}_{row['arm']}"
    fm_path = final_models.get(key, {}).get('checkpoint_path')
    assert fm_path == row['final_checkpoint_path'], \
        f'Checkpoint path mismatch for {key}: final_models.json={fm_path} vs CSV={row["final_checkpoint_path"]}'

print('Inpainted-arm registry (6 models):')
print(inpainted_registry[['architecture', 'mode', 'dropout', 'unfreeze_category', 'final_checkpoint_path']].to_string(index=False))

### 3.2 Model Loader

Mirrors N04's `load_eval_model` exactly (not imported — it's notebook-local code in N04, not a `src/` module). `.eval()` is set unconditionally, same rationale as N04: eliminates the possibility of a silently-wrong forward pass.

In [ ]:
def load_gradcam_model(architecture, mode, dropout, unfreeze_category, checkpoint_path):
    unfreeze_category = unfreeze_category if pd.notna(unfreeze_category) else None
    model = build_model(architecture, mode, dropout, unfreeze_category=unfreeze_category).to(device)
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()
    return model

### 3.3 Target Layer Definitions

Derived directly from `src/models.py`'s `freeze_*` functions, which name real module attributes that must exist for freezing to have worked during training:

- **ResNet50**: `freeze_resnet50` unfreezes `model.layer4` for `last_1_block`. The target layer is `model.layer4[-1]` — the final Bottleneck block, standard GradCAM practice for ResNets.
- **EfficientNet-B0**: `freeze_efficientnet_b0` references `model.features[7]` and `model.features[8]` as the last two blocks. The target layer is `model.features[-1]` (`= model.features[8]`) — the final `Conv2dNormActivation` before pooling.
- **Swin-Tiny**: `freeze_swin_t` references `model.features[7]` as the last stage. Combined with the Section-4.1 structural inspection from N03 (`features[7]: Sequential`, containing `SwinTransformerBlock`s with `norm1`/`attn`/`norm2`/`mlp` submodules — standard torchvision Swin block structure), the target layer is `model.features[7][-1].norm2`, matching the `pytorch-grad-cam` library's own Swin usage example. Only a channel-order permute is needed, considering the `SwinTransformerBlock` outputs are already spatial.

In [ ]:
def get_target_layer(model, architecture):
    if architecture == 'resnet50':
        return model.layer4[-1]
    elif architecture == 'efficientnet_b0':
        return model.features[-1]
    elif architecture == 'swin_tiny':
        return model.features[7][-1].norm2
    else:
        raise ValueError(f'Unknown architecture: {architecture}')


def swin_reshape_transform(tensor):
    """
    torchvision's SwinTransformerBlock outputs (batch, H, W, C) -- already spatial,
    unlike timm's flattened (batch, N, C) that the library's official example assumes.
    Only a channel-order permute is needed, confirmed against the real (1, 7, 7, 768)
    shape observed in 3.4 -- not a reshape/unflatten.
    """
    return tensor.permute(0, 3, 1, 2)


RESHAPE_TRANSFORMS = {
    'resnet50': None,
    'efficientnet_b0': None,
    'swin_tiny': swin_reshape_transform,
}

### 3.4 Structural Verification: Forward Hook Shape Check

Loads one TL model per architecture, attaches a forward hook to the target layer, runs a single real forward pass, and confirms the raw activation shape and, for Swin, the shape after `swin_reshape_transform` matches what GradCAM++ requires: `(batch, C, 7, 7)`.

In [ ]:
def swin_reshape_transform(tensor):
    """
    torchvision's SwinTransformerBlock outputs (batch, H, W, C) -- already spatial,
    unlike timm's flattened (batch, N, C) that the library's official example assumes.
    Only a channel-order permute is needed, confirmed against the real (1, 7, 7, 768)
    shape observed in 3.4 -- not a reshape/unflatten.
    """
    return tensor.permute(0, 3, 1, 2)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sample_row = unique_images[unique_images['dataset'] == 'BUSI'].sample(1, random_state=SEED).iloc[0]
sample_image_arr = load_final_image(sample_row['dataset'], sample_row['image_name'], sample_row['class_name'])
sample_pil = Image.fromarray(sample_image_arr)

transform = build_transform('eval', IMAGENET_MEAN, IMAGENET_STD)
sample_tensor = transform(sample_pil).unsqueeze(0).to(device)

EXPECTED_RAW_SHAPE = {
    'resnet50':        (1, 2048, 7, 7),
    'efficientnet_b0': (1, 1280, 7, 7),
    'swin_tiny':        (1, 7, 7, 768),
}
EXPECTED_FINAL_SHAPE = (1, None, 7, 7)

loaded_tl_models = {}

for architecture in ['resnet50', 'efficientnet_b0', 'swin_tiny']:
    row = inpainted_registry[
        (inpainted_registry['architecture'] == architecture) & (inpainted_registry['mode'] == 'TL')
    ].iloc[0]

    model = load_gradcam_model(
        architecture, row['mode'], row['dropout'], row['unfreeze_category'], row['final_checkpoint_path']
    )
    loaded_tl_models[architecture] = model

    target_layer = get_target_layer(model, architecture)
    captured = {}

    def make_hook(arch):
        def hook(module, inp, out):
            captured['raw'] = out.detach()
        return hook

    handle = target_layer.register_forward_hook(make_hook(architecture))
    with torch.inference_mode():
        _ = model(sample_tensor)
    handle.remove()

    raw_shape = tuple(captured['raw'].shape)
    assert raw_shape == EXPECTED_RAW_SHAPE[architecture], \
        f'{architecture}: raw target-layer shape {raw_shape}, expected {EXPECTED_RAW_SHAPE[architecture]}'

    reshape_fn = RESHAPE_TRANSFORMS[architecture]
    final_activation = reshape_fn(captured['raw']) if reshape_fn else captured['raw']
    final_shape = tuple(final_activation.shape)
    assert final_shape[0] == 1 and final_shape[2] == 7 and final_shape[3] == 7, \
        f'{architecture}: final activation shape {final_shape}, expected (1, C, 7, 7)'

    print(f'{architecture}: raw={raw_shape}  ->  final={final_shape}  [OK]')

print('\nSection 3.4 structural verification passed for all 3 architectures.')

### 3.5 End-to-End GradCAM++ Smoke Test

Builds a real `GradCAMPlusPlus` instance for each of the 3 already-loaded TL models and generates one real heatmap for the predicted class, confirming: no runtime errors from the library's own internals, output shape `(224, 224)`, and values in `[0, 1]`.

In [ ]:
input_np_float = np.float32(sample_image_arr) / 255.0  # for show_cam_on_image overlay

for architecture, model in loaded_tl_models.items():
    target_layer = get_target_layer(model, architecture)
    reshape_fn = RESHAPE_TRANSFORMS[architecture]

    cam = GradCAMPlusPlus(model=model, target_layers=[target_layer], reshape_transform=reshape_fn)

    with torch.inference_mode():
        logits = model(sample_tensor)
        pred_class = logits.argmax(dim=1).item()

    targets = [ClassifierOutputTarget(pred_class)]
    grayscale_cam = cam(input_tensor=sample_tensor, targets=targets)[0]  # gradients needed -- no inference_mode here

    assert grayscale_cam.shape == (224, 224), \
        f'{architecture}: heatmap shape {grayscale_cam.shape}, expected (224, 224)'
    assert 0.0 <= grayscale_cam.min() and grayscale_cam.max() <= 1.0 + 1e-5, \
        f'{architecture}: heatmap values out of [0, 1] range (min={grayscale_cam.min()}, max={grayscale_cam.max()})'

    overlay = show_cam_on_image(input_np_float, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    axes[0].imshow(sample_image_arr); axes[0].set_title('Original', fontsize=9)
    axes[1].imshow(grayscale_cam, cmap='jet'); axes[1].set_title('GradCAM++ raw', fontsize=9)
    axes[2].imshow(overlay); axes[2].set_title('Overlay', fontsize=9)
    for ax in axes:
        ax.axis('off')
    fig.suptitle(f'{architecture} (TL) -- predicted class idx {pred_class}', fontsize=10)
    plt.tight_layout()
    plt.show()

    del cam
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print('\nSection 3.5 smoke test passed: real GradCAM++ output for all 3 architectures.')

# Cleanup -- these were verification-only models, Section 5 loads the full 6 fresh.
del loaded_tl_models
if device.type == 'cuda':
    torch.cuda.empty_cache()

## 4. Image Selection

Builds a deterministic, purposively-sampled stratified subset of images across all four strata, using all 6 inpainted-arm models' predictions jointly. Produces two outputs:

- `stratum_membership`: long-format table, one row per (image, stratum) pair; an image can belong to more than one stratum.
- `selected_images`: deduplicated unique image list, used to drive GradCAM++ generation in Section 5. Each unique image is processed once, regardless of how many strata it satisfies.

Selection criteria are stated in full below.

### 4.1 Per-Image Correctness Matrix

Pivots the 6 inpainted-arm models' predictions into one row per (dataset, image), with one boolean column per model (`{architecture}_{mode}`), plus the exact set of which models failed on that image; needed for Stratum 3's idiosyncratic/systemic split.

In [ ]:
inpainted_preds['model_id'] = inpainted_preds['architecture'] + '_' + inpainted_preds['mode']
inpainted_preds['correct']  = inpainted_preds['true_label'] == inpainted_preds['pred_label']

dup_check = inpainted_preds.duplicated(subset=['eval_data', 'image_name', 'model_id']).sum()
assert dup_check == 0, f'{dup_check} duplicate (eval_data, image_name, model_id) rows -- investigate before pivoting.'

model_ids = sorted(inpainted_preds['model_id'].unique())
assert len(model_ids) == 6, f'Expected 6 model_ids, got {len(model_ids)}: {model_ids}'

correctness_wide = inpainted_preds.pivot_table(
    index=['eval_data', 'image_name', 'true_label'],
    columns='model_id', values='correct', aggfunc='first'
).reset_index()

assert correctness_wide[model_ids].isna().sum().sum() == 0, 'NaN found in correctness matrix -- some (image, model) pair missing.'

correctness_wide['dataset']    = correctness_wide['eval_data'].map(EVAL_DATA_TO_DATASET)
correctness_wide['class_name'] = correctness_wide['true_label'].map(IDX_TO_CLASS)
correctness_wide['n_correct']  = correctness_wide[model_ids].sum(axis=1)
correctness_wide['all_correct'] = correctness_wide['n_correct'] == 6
correctness_wide['failing_models'] = correctness_wide.apply(
    lambda r: tuple(m for m in model_ids if not r[m]), axis=1
)
correctness_wide['n_failing'] = correctness_wide['failing_models'].apply(len)

print(f'correctness_wide: {correctness_wide.shape[0]} unique images across all eval sets.')
print(f'Model columns: {model_ids}')

### 4.2 Stratum 1: BUSI-Test, Consensus-Correct

All 6 models agree and are correct. Target: 3 per class, capped at pool size.

In [ ]:
stratum1_pool = correctness_wide[(correctness_wide['eval_data'] == 'busi_test') & (correctness_wide['all_correct'])]

stratum1_parts = []
for cls in ['benign', 'malignant', 'normal']:
    candidates = stratum1_pool[stratum1_pool['class_name'] == cls]
    n = min(3, len(candidates))
    sampled = candidates.sample(n=n, random_state=SEED)
    stratum1_parts.append(sampled)
    print(f'Stratum 1 / busi_test / {cls}: pool={len(candidates)}, selected={n}')

stratum1_df = pd.concat(stratum1_parts, ignore_index=True)

### 4.3 Stratum 2: External, Consensus-Correct

All 6 models agree and are correct, BrEaST + BUS-UCLM. Target: 3 per class per dataset.
Given N04's findings, the external-malignant cell is the one to watch.

In [ ]:
stratum2_parts = []
for dataset, eval_data in [('BrEaST', 'breast'), ('BUS-UCLM', 'bus_uclm')]:
    pool = correctness_wide[(correctness_wide['eval_data'] == eval_data) & (correctness_wide['all_correct'])]
    for cls in ['benign', 'malignant', 'normal']:
        candidates = pool[pool['class_name'] == cls]
        n = min(3, len(candidates))
        sampled = candidates.sample(n=n, random_state=SEED)
        stratum2_parts.append(sampled)
        flag = '  <-- external malignant consensus-correct' if (cls == 'malignant') else ''
        print(f'Stratum 2 / {eval_data} / {cls}: pool={len(candidates)}, selected={n}{flag}')

stratum2_df = pd.concat(stratum2_parts, ignore_index=True)

### 4.4 Stratum 3: External Malignant, Misclassified (Idiosyncratic + Systemic)

Two tiers per dataset (BrEaST, BUS-UCLM), malignant class only:

- **Tier A (idiosyncratic)**: up to one image per model where *only that model* failed (`failing_models == (model_id,)`), covering as many distinct models as have such a case.
- **Tier B (systemic)**: fills the remaining budget (9 total per dataset) with images where 2+ models failed together, sampled deterministically, excluding Tier A images.

If Tier A returns nothing for a given model, that is reported explicitly -- it means failure was never architecture-specific for that model on this dataset, which is itself informative.

In [ ]:
STRATUM3_BUDGET = 9

stratum3_parts = []
for dataset, eval_data in [('BrEaST', 'breast'), ('BUS-UCLM', 'bus_uclm')]:
    pool = correctness_wide[
        (correctness_wide['eval_data'] == eval_data) &
        (correctness_wide['class_name'] == 'malignant') &
        (correctness_wide['n_failing'] >= 1)
    ]
    print(f'\nStratum 3 / {eval_data}: total misclassified-by-at-least-one-model pool = {len(pool)}')

    tier_a_parts = []
    coverage = {}
    for model_id in model_ids:
        idiosyncratic = pool[pool['failing_models'] == (model_id,)]
        coverage[model_id] = len(idiosyncratic)
        if len(idiosyncratic) > 0:
            tier_a_parts.append(idiosyncratic.sample(n=1, random_state=SEED))

    tier_a_df = pd.concat(tier_a_parts, ignore_index=True) if tier_a_parts else pool.iloc[0:0].copy()
    assert not tier_a_df.duplicated(subset=['image_name']).any(), 'Unexpected duplicate in Tier A -- investigate.'

    print(f'  Tier A idiosyncratic-failure coverage per model: {coverage}')
    print(f'  Tier A selected: {len(tier_a_df)}')

    remaining_budget = STRATUM3_BUDGET - len(tier_a_df)
    tier_b_pool = pool[(pool['n_failing'] >= 2) & (~pool['image_name'].isin(tier_a_df['image_name']))]
    n_b = min(max(remaining_budget, 0), len(tier_b_pool))
    tier_b_df = tier_b_pool.sample(n=n_b, random_state=SEED)

    print(f'  Tier B systemic-failure pool: {len(tier_b_pool)}, budget remaining: {remaining_budget}, selected: {n_b}')

    stratum3_parts.append(tier_a_df)
    stratum3_parts.append(tier_b_df)

stratum3_df = pd.concat(stratum3_parts, ignore_index=True)

### 4.5 Stratum 4: BrEaST `has_other_lesion`

Unconditional: all 7 flagged images, regardless of any model's prediction.

In [ ]:
breast_other_lesion_names = df_clean[
    (df_clean['dataset'] == 'BrEaST') & (df_clean['has_other_lesion'] == True)
]['image_name'].tolist()
assert len(breast_other_lesion_names) == 7, f'Expected 7 BrEaST has_other_lesion images, got {len(breast_other_lesion_names)}.'

stratum4_df = correctness_wide[
    (correctness_wide['eval_data'] == 'breast') &
    (correctness_wide['image_name'].isin(breast_other_lesion_names))
].copy()

assert len(stratum4_df) == 7, f'Expected all 7 has_other_lesion images matched in predictions, got {len(stratum4_df)}.'
print(f'Stratum 4: {len(stratum4_df)}/7 selected (unconditional).')

### 4.6 Combine, Deduplicate & Save

In [ ]:
STRATUM_LABELS = {
    1: 'busi_consensus_correct',
    2: 'external_consensus_correct',
    3: 'external_malignant_misclassified',
    4: 'breast_other_lesion',
}

def tag_stratum(df, stratum_id):
    out = df[['dataset', 'eval_data', 'image_name', 'class_name']].copy()
    out['stratum_id'] = stratum_id
    out['stratum_label'] = STRATUM_LABELS[stratum_id]
    return out

stratum_membership = pd.concat([
    tag_stratum(stratum1_df, 1),
    tag_stratum(stratum2_df, 2),
    tag_stratum(stratum3_df, 3),
    tag_stratum(stratum4_df, 4),
], ignore_index=True)

selected_images = stratum_membership[['dataset', 'eval_data', 'image_name', 'class_name']].drop_duplicates().reset_index(drop=True)

print('=== Stratum membership (pre-dedup) ===')
print(stratum_membership.groupby('stratum_label')['image_name'].count().to_string())
print(f'\n=== Unique images selected (post-dedup): {len(selected_images)} ===')
print(selected_images.groupby(['dataset', 'class_name'])['image_name'].count().to_string())
print(f'\nTotal GradCAM++ heatmaps to generate: {len(selected_images)} images x 6 models = {len(selected_images) * 6}')

stratum_membership.to_csv(GRADCAM_DIR / 'stratum_membership.csv', index=False)
selected_images.to_csv(GRADCAM_DIR / 'selected_images.csv', index=False)
print(f'\nSaved: {GRADCAM_DIR / "stratum_membership.csv"}')
print(f'Saved: {GRADCAM_DIR / "selected_images.csv"}')

## 5. GradCAM++ Generation

Generates raw GradCAM++ heatmaps for all 50 selected images across all 6 inpainted-arm models. For each (image, model) pair: one heatmap for the **predicted** class always, plus a second heatmap for the **true** class when the model misclassified that specific image. This applies uniformly across all 4 strata by construction (not assumed per-stratum).

Output: `gradcam_manifest.csv` (one row per generated heatmap, full metadata) and `gradcam_heatmaps.npz` (the raw 224x224 float32 arrays, keyed by `heatmap_id`). Respects `FORCE_REGENERATE` -- skips generation entirely if both files already exist.

### 5.1 Scratch-Mode Normalisation Stats

TL models use ImageNet stats throughout. Scratch models were trained with dataset-specific stats, computed once in N04 and saved to `busi_scratch_normalization_stats.json`. Only the `'inpainted'` entry is relevant here.

In [ ]:
scratch_stats_path = RESULTS / 'busi_scratch_normalization_stats.json'
assert scratch_stats_path.exists(), f'{scratch_stats_path} not found -- was N04 run to completion?'

scratch_stats_all = json.load(open(scratch_stats_path))
assert 'inpainted' in scratch_stats_all, "'inpainted' arm missing from scratch stats file."

scratch_mean, scratch_std = scratch_stats_all['inpainted']['mean'], scratch_stats_all['inpainted']['std']
print(f'Scratch-mode (inpainted arm) normalisation: mean={scratch_mean}, std={scratch_std}')

def get_normalization(mode):
    return (IMAGENET_MEAN, IMAGENET_STD) if mode == 'TL' else (scratch_mean, scratch_std)

### 5.2 Generation Manifest

Expands the 50 selected images x 6 models into long format, one row per heatmap to be generated. Sourced entirely from `predictions.csv` (already-known `pred_label`/`true_label`) joined against `selected_images` from Section 4 -- no model inference needed to build this plan.

In [ ]:
preds_for_selected = selected_images.merge(
    inpainted_preds[['eval_data', 'image_name', 'architecture', 'mode', 'model_id', 'true_label', 'pred_label', 'correct']],
    on=['eval_data', 'image_name'], how='left'
)
assert len(preds_for_selected) == len(selected_images) * 6, \
    f'Expected {len(selected_images) * 6} rows, got {len(preds_for_selected)} -- join produced unexpected row count.'
assert preds_for_selected['model_id'].isna().sum() == 0, 'Some selected images missing predictions for at least one model.'

manifest_rows = []
heatmap_id = 0
for _, row in preds_for_selected.iterrows():
    targets = [(row['pred_label'], True, bool(row['pred_label'] == row['true_label']))]
    if not row['correct']:
        targets.append((row['true_label'], False, True))

    for target_class, is_predicted, is_true in targets:
        manifest_rows.append({
            'heatmap_id':    heatmap_id,
            'dataset':       row['dataset'],
            'eval_data':     row['eval_data'],
            'image_name':    row['image_name'],
            'class_name':    row['class_name'],
            'architecture':  row['architecture'],
            'mode':          row['mode'],
            'model_id':      row['model_id'],
            'true_label':    row['true_label'],
            'pred_label':    row['pred_label'],
            'correct':       row['correct'],
            'target_class':  target_class,
            'target_class_name': IDX_TO_CLASS[target_class],
            'is_predicted':  is_predicted,
            'is_true':       is_true,
        })
        heatmap_id += 1

gradcam_manifest = pd.DataFrame(manifest_rows)

n_misclassified_rows = (~preds_for_selected['correct']).sum()
expected_total = len(preds_for_selected) + n_misclassified_rows
assert len(gradcam_manifest) == expected_total, \
    f'Manifest has {len(gradcam_manifest)} rows, expected {expected_total} (300 base + {n_misclassified_rows} extra true-class heatmaps).'

print(f'Generation manifest: {len(gradcam_manifest)} heatmaps planned')
print(f'  Base (predicted-class, one per image-model): {len(preds_for_selected)}')
print(f'  Extra (true-class, misclassified only):       {n_misclassified_rows}')
print(gradcam_manifest.groupby(['architecture', 'mode'])['heatmap_id'].count().to_string())

### 5.3 Generation Loop

Grouped by `(architecture, mode)` so each of the 6 models is loaded exactly once. Within each group, one real forward+backward pass per manifest row. `cam(...)` is deliberately outside any `inference_mode()` context -- gradients are required. Progress printed every 25 heatmaps.

In [ ]:
manifest_path = GRADCAM_DIR / 'gradcam_manifest.csv'
npz_path = GRADCAM_DIR / 'gradcam_heatmaps.npz'

if manifest_path.exists() and npz_path.exists() and not FORCE_REGENERATE:
    print(f'Found existing results at {GRADCAM_DIR} -- loading instead of regenerating.')
    print('(Set FORCE_REGENERATE = True in Section 1 to force a fresh run.)')
    gradcam_manifest = pd.read_csv(manifest_path)
    heatmaps_npz = np.load(npz_path)
    heatmaps = {int(k[1:]): heatmaps_npz[k] for k in heatmaps_npz.files}
    print(f'Loaded gradcam_manifest: {gradcam_manifest.shape}, {len(heatmaps)} heatmap arrays')

else:
    heatmaps = {}
    n_generated = 0

    for (architecture, mode), group in gradcam_manifest.groupby(['architecture', 'mode']):
        model_row = inpainted_registry[
            (inpainted_registry['architecture'] == architecture) & (inpainted_registry['mode'] == mode)
        ].iloc[0]

        print(f'\nLoading {architecture}/{mode}...')
        model = load_gradcam_model(
            architecture, mode, model_row['dropout'], model_row['unfreeze_category'], model_row['final_checkpoint_path']
        )

        target_layer = get_target_layer(model, architecture)
        reshape_fn = RESHAPE_TRANSFORMS[architecture]
        cam = GradCAMPlusPlus(model=model, target_layers=[target_layer], reshape_transform=reshape_fn)

        mean, std = get_normalization(mode)
        transform = build_transform('eval', mean, std)

        for _, row in group.iterrows():
            image_arr = load_final_image(row['dataset'], row['image_name'], row['class_name'])
            input_tensor = transform(Image.fromarray(image_arr)).unsqueeze(0).to(device)

            targets = [ClassifierOutputTarget(int(row['target_class']))]
            grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

            assert grayscale_cam.shape == (224, 224), \
                f'heatmap_id {row["heatmap_id"]}: shape {grayscale_cam.shape}, expected (224, 224)'
            assert 0.0 <= grayscale_cam.min() and grayscale_cam.max() <= 1.0 + 1e-5, \
                f'heatmap_id {row["heatmap_id"]}: values out of [0,1] range'

            heatmaps[int(row['heatmap_id'])] = grayscale_cam.astype(np.float32)
            n_generated += 1

            if n_generated % 25 == 0:
                print(f'  ...{n_generated}/{len(gradcam_manifest)} heatmaps generated')

        del cam, model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    print(f'\nDone. Generated: {n_generated}/{len(gradcam_manifest)}')

    gradcam_manifest.to_csv(manifest_path, index=False)
    np.savez(npz_path, **{f'h{k}': v for k, v in heatmaps.items()})
    print(f'Saved: {manifest_path}')
    print(f'Saved: {npz_path}')

### 5.4 Completeness Verification

Kept permanently, re-run every session. Confirms every manifest row has a corresponding heatmap array, no array is degenerate (all-zero or all-one, a common silent-failure signature for GradCAM when the wrong layer or a dead gradient path is used), and row counts match expectations.

In [ ]:
assert set(gradcam_manifest['heatmap_id']) == set(heatmaps.keys()), \
    'Mismatch between manifest heatmap_ids and saved heatmap arrays.'

degenerate = []
for hid, arr in heatmaps.items():
    if arr.std() < 1e-6:
        degenerate.append(hid)

if degenerate:
    print(f'WARNING: {len(degenerate)} degenerate (near-constant) heatmap(s): {degenerate[:10]}{"..." if len(degenerate) > 10 else ""}')
    print('This does not necessarily mean a bug -- but investigate before treating these as valid results.')
else:
    print('No degenerate heatmaps found.')

per_model_counts = gradcam_manifest.groupby(['architecture', 'mode'])['heatmap_id'].count()
print('\nHeatmaps per model:')
print(per_model_counts.to_string())

print(f'\nTotal: {len(gradcam_manifest)} heatmaps, {len(heatmaps)} arrays -- verified consistent.')

## 6. Heatmap Visualisation & Alignment Verification

Builds a reusable render function, then verifies, visually and with a coarse numeric check, that heatmaps and masks actually align with each other on the same image. Sections 2 and 5 each verified their own half (mask-to-image, heatmap shape/range); neither has confirmed the two halves agree with each other. This section closes that gap before Section 7's IoU computation
depends on it.

### 6.1 Render Function

Three panels per heatmap: original image, GradCAM++ overlay, and the overlay with the mask boundary drawn as a contour on top -- the third panel is the actual alignment check. `None` for images with no mask (normal class, or missing file), consistent with Section 2's `found=False` handling.

In [ ]:
def render_heatmap_id(heatmap_id, show=True):
    row = gradcam_manifest[gradcam_manifest['heatmap_id'] == heatmap_id].iloc[0]
    image_arr = load_final_image(row['dataset'], row['image_name'], row['class_name'])
    heatmap = heatmaps[heatmap_id]

    input_np_float = np.float32(image_arr) / 255.0
    overlay = show_cam_on_image(input_np_float, heatmap, use_rgb=True)

    mask_result = get_lesion_mask(row['dataset'], row['image_name'], row['class_name'])
    mask = mask_result['mask'] if mask_result['found'] else None

    overlay_with_contour = overlay.copy()
    if mask is not None:
        mask_uint8 = (mask * 255).astype(np.uint8)
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay_with_contour, contours, -1, (255, 255, 255), 2)

    target_desc = row['target_class_name']
    target_desc += ' (predicted)' if row['is_predicted'] else ' (true, misclassified)'

    fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
    axes[0].imshow(image_arr)
    axes[0].set_title(f"{row['dataset']}/{row['image_name']}\ntrue={row['class_name']}", fontsize=8)
    axes[1].imshow(overlay)
    axes[1].set_title(f"{row['architecture']}/{row['mode']}\ntarget={target_desc}", fontsize=8)
    axes[2].imshow(overlay_with_contour)
    axes[2].set_title(f"Heatmap + mask boundary\n({'mask found' if mask is not None else 'no mask'})", fontsize=8)

    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    if show:
        plt.show()
    return fig, mask

### 6.2 Verification on Real Samples

Six real generated heatmaps, deliberately covering: a masked BUSI case, a masked external case, a normal-class case (no mask expected), a misclassified case (true-class target), and at least one Swin-Tiny sample -- re-confirming the 3.4 reshape-transform fix holds inside the full pipeline, not just the isolated smoke test. A coarse numeric check accompanies each masked sample: heatmap "hot" pixels and the mask must actually overlap at all -- not a real IoU value, just a gross-misalignment tripwire.

In [ ]:
verification_targets = []

# Masked BUSI, predicted class
c1 = gradcam_manifest[(gradcam_manifest['dataset'] == 'BUSI') & (gradcam_manifest['class_name'] != 'normal') & (gradcam_manifest['is_predicted'])]
verification_targets.append(('BUSI masked case', c1.sample(1, random_state=SEED).iloc[0]['heatmap_id']))

# Masked external case (BrEaST or BUS-UCLM), predicted class
c2 = gradcam_manifest[(gradcam_manifest['dataset'] != 'BUSI') & (gradcam_manifest['class_name'] != 'normal') & (gradcam_manifest['is_predicted'])]
verification_targets.append(('External masked case', c2.sample(1, random_state=SEED).iloc[0]['heatmap_id']))

# Normal class, no mask expected
c3 = gradcam_manifest[gradcam_manifest['class_name'] == 'normal']
verification_targets.append(('Normal (no mask expected)', c3.sample(1, random_state=SEED).iloc[0]['heatmap_id']))

# Misclassified, true-class target
c4 = gradcam_manifest[~gradcam_manifest['is_predicted']]
assert len(c4) > 0, 'No true-class (misclassified) heatmaps found -- Section 5 manifest looks wrong.'
verification_targets.append(('Misclassified, true-class target', c4.sample(1, random_state=SEED).iloc[0]['heatmap_id']))

# Swin-Tiny, masked case -- re-confirms reshape_transform in the full pipeline
c5 = gradcam_manifest[(gradcam_manifest['architecture'] == 'swin_tiny') & (gradcam_manifest['class_name'] != 'normal')]
verification_targets.append(('Swin-Tiny masked case', c5.sample(1, random_state=SEED).iloc[0]['heatmap_id']))

# One more external, different architecture, for spread
c6 = gradcam_manifest[(gradcam_manifest['architecture'] == 'resnet50') & (gradcam_manifest['dataset'] == 'BUS-UCLM') & (gradcam_manifest['class_name'] != 'normal')]
verification_targets.append(('BUS-UCLM ResNet50 case', c6.sample(1, random_state=SEED).iloc[0]['heatmap_id']))

for label, hid in verification_targets:
    print(f'=== {label} (heatmap_id={hid}) ===')
    fig, mask = render_heatmap_id(hid)

    if mask is not None:
        hot = heatmaps[hid] > 0.5
        overlap = np.logical_and(hot, mask.astype(bool)).sum()
        assert overlap > 0, (
            f'heatmap_id {hid}: ZERO overlap between hot heatmap pixels and mask -- '
            f'gross misalignment, investigate before trusting Section 7.'
        )
        print(f'  Coarse overlap check: {overlap} overlapping pixels (hot pixels: {hot.sum()}, mask pixels: {mask.sum()}) -- PASS')
    else:
        print('  No mask for this image (expected for normal class) -- visual check only.')
    print()

print('Section 6.2 verification complete: all 6 samples rendered, no gross misalignment detected.')

## 7. Quantitative Localisation

Computes two threshold-free/threshold-light metrics per heatmap, wherever a mask exists:

- **EBPG (Energy-Based Pointing Game)**: fraction of the heatmap's total continuous "energy" falling inside the lesion mask. No binarisation, no threshold decision.
- **Otsu-IoU**: heatmap binarised via Otsu's method (automatic, data-driven per-image threshold, not hand-picked), then standard IoU against the mask. Secondary/companion metric.

For misclassified images, both metrics are already available for **both** the predicted-class and true-class heatmaps directly from Section 5's manifest -- no restructuring needed, since `is_predicted`/`is_true` already tag which heatmap is which. For correctly classified images, predicted and true class are the same heatmap, so this collapses naturally to one value.

Reporting is descriptive only (mean/median/IQR/n) -- no significance testing, given cell sizes in the single digits in several strata.

### 7.1 Metric Functions

In [ ]:
def compute_ebpg(heatmap, mask):
    total_energy = heatmap.sum()
    if total_energy <= 0:
        return np.nan
    return float((heatmap * mask).sum() / total_energy)


def compute_otsu_iou(heatmap, mask):
    heatmap_uint8 = (heatmap * 255).astype(np.uint8)
    _, binarized = cv2.threshold(heatmap_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    binarized = (binarized > 0).astype(np.uint8)
    mask_bin = mask.astype(np.uint8)

    intersection = np.logical_and(binarized, mask_bin).sum()
    union = np.logical_or(binarized, mask_bin).sum()
    if union == 0:
        return np.nan
    return float(intersection / union)

### 7.2 Compute Localisation Metrics

Masks are cached per unique `(dataset, image_name, class_name)`; several heatmaps (up to 6 models x up to 2 targets) share the same underlying image and mask, so this avoids redundant raw-file reloading. Heatmaps with no mask (normal class primarily) are skipped and counted, not silently dropped.

In [ ]:
localization_path = GRADCAM_DIR / 'gradcam_localization.csv'

mask_cache = {}
def get_cached_mask(dataset, image_name, class_name):
    key = (dataset, image_name, class_name)
    if key not in mask_cache:
        result = get_lesion_mask(dataset, image_name, class_name)
        mask_cache[key] = result['mask'] if result['found'] else None
    return mask_cache[key]


if localization_path.exists() and not FORCE_REGENERATE:
    print(f'Found existing results at {localization_path} -- loading instead of recomputing.')
    localization_df = pd.read_csv(localization_path)
else:
    records = []
    n_no_mask = 0

    for _, row in gradcam_manifest.iterrows():
        mask = get_cached_mask(row['dataset'], row['image_name'], row['class_name'])
        if mask is None:
            n_no_mask += 1
            continue

        heatmap = heatmaps[row['heatmap_id']]
        record = row.to_dict()
        record['ebpg'] = compute_ebpg(heatmap, mask)
        record['otsu_iou'] = compute_otsu_iou(heatmap, mask)
        records.append(record)

    localization_df = pd.DataFrame(records)
    print(f'Computed localisation metrics for {len(localization_df)}/{len(gradcam_manifest)} heatmaps '
          f'({n_no_mask} skipped -- no mask, expected for normal-class and unmasked BrEaST-normal images).')

    localization_df.to_csv(localization_path, index=False)
    print(f'Saved: {localization_path}')

# Sanity check
assert localization_df['ebpg'].between(0, 1, inclusive='both').all(), 'EBPG value out of [0,1] range.'
assert localization_df['otsu_iou'].between(0, 1, inclusive='both').all(), 'Otsu-IoU value out of [0,1] range.'
print(f'\nRange check passed. {len(localization_df)} rows with valid metrics.')

### 7.3 Descriptive Summary: By Architecture / Mode

Uses `is_predicted` rows only (full coverage: every heatmap has a predicted-class target).

In [ ]:
by_model = localization_df[localization_df['is_predicted']].groupby(['architecture', 'mode']).agg(
    n=('ebpg', 'count'),
    ebpg_mean=('ebpg', 'mean'), ebpg_median=('ebpg', 'median'),
    iou_mean=('otsu_iou', 'mean'), iou_median=('otsu_iou', 'median'),
).round(3)

print('Localisation quality by model (predicted-class heatmaps):')
print(by_model.to_string())

### 7.4 Descriptive Summary: By Dataset x Correctness

In [ ]:
by_dataset_correct = localization_df[localization_df['is_predicted']].groupby(['dataset', 'correct']).agg(
    n=('ebpg', 'count'),
    ebpg_mean=('ebpg', 'mean'), ebpg_median=('ebpg', 'median'),
    iou_mean=('otsu_iou', 'mean'), iou_median=('otsu_iou', 'median'),
).round(3)

print('Localisation quality by dataset and correctness (predicted-class heatmaps):')
print(by_dataset_correct.to_string())

### 7.5 The Key Comparison: Predicted vs. True Localisation on Misclassified Images

This is the table that speaks directly to the pattern in Section 6 (EfficientNet-B0's near-perfect true-class localisation despite misclassifying). High true-class EBPG/IoU alongside a wrong prediction is direct quantitative evidence that failure is not always an attention failure.

In [ ]:
misclassified_only = localization_df[~localization_df['correct']]

pred_vs_true = misclassified_only.groupby(['dataset', 'is_predicted']).agg(
    n=('ebpg', 'count'),
    ebpg_mean=('ebpg', 'mean'), ebpg_median=('ebpg', 'median'),
    iou_mean=('otsu_iou', 'mean'), iou_median=('otsu_iou', 'median'),
).round(3)

pred_vs_true.index = pred_vs_true.index.set_levels(
    ['true-class (missed evidence)', 'predicted-class (wrong evidence)'], level='is_predicted'
)

print('Misclassified images only -- predicted-class vs. true-class heatmap localisation:')
print(pred_vs_true.to_string())

# Paired view: same image, both heatmaps, side by side
paired = misclassified_only.pivot_table(
    index=['dataset', 'image_name', 'architecture', 'mode'],
    columns='is_predicted', values=['ebpg', 'otsu_iou']
)
paired.columns = ['ebpg_true', 'ebpg_pred', 'iou_true', 'iou_pred']
paired = paired.reset_index()
paired['ebpg_gap'] = paired['ebpg_true'] - paired['ebpg_pred']

print(f'\nPer-image paired comparison (n={len(paired)} misclassified image-model pairs):')
print(paired[['dataset', 'architecture', 'mode', 'ebpg_true', 'ebpg_pred', 'ebpg_gap']].sort_values('ebpg_gap', ascending=False).head(10).to_string(index=False))

#### Additional Checks

In [ ]:
print('Full distribution of ebpg_gap (true-class EBPG minus predicted-class EBPG), n=61:')
print(paired['ebpg_gap'].describe().round(3).to_string())

print('\nBottom 10 -- predicted-class localises BETTER than true-class (the opposite pattern):')
print(paired[['dataset', 'architecture', 'mode', 'ebpg_true', 'ebpg_pred', 'ebpg_gap']]
      .sort_values('ebpg_gap', ascending=True).head(10).to_string(index=False))

In [ ]:
gap_by_mode = paired.groupby('mode')['ebpg_gap'].agg(['count', 'mean', 'median', 'std']).round(3)
print('EBPG gap (true - predicted) by training mode:')
print(gap_by_mode.to_string())

paired['notable_negative'] = paired['ebpg_gap'] < -0.10
print('\nCount of "predicted-class localises notably better than true-class" (gap < -0.10), by mode:')
print(pd.crosstab(paired['mode'], paired['notable_negative']))

## 8. Internal vs. External Attention Comparison

Two parts: (1) a direct internal-vs-external localisation comparison using the existing `localization_df`, explicitly caveated by Section 4's own selection design; (2) a padding-energy density check, testing the faint edge-activation pattern observed qualitatively in Sections 3 and 6, computed as an energy-per-pixel density ratio (padding vs content) to avoid confounding with
each dataset's differing amount of physical letterbox padding.

This section is exploratory and descriptive, complementary to -- not a replacement for -- N04's ablation-arm Clever Hans investigation (verdict: inconclusive).

### 8.1 Internal vs. External Localisation

Reuses `localization_df` from Section 7. `internal` = BUSI-test only; `external` = BrEaST + BUS-UCLM combined.

In [ ]:
localization_df['region'] = localization_df['dataset'].apply(lambda d: 'internal' if d == 'BUSI' else 'external')

region_summary = localization_df[localization_df['is_predicted']].groupby(['region', 'correct']).agg(
    n=('ebpg', 'count'),
    ebpg_mean=('ebpg', 'mean'), ebpg_median=('ebpg', 'median'),
    iou_mean=('otsu_iou', 'mean'), iou_median=('otsu_iou', 'median'),
).round(3)

print('Localisation quality: internal (BUSI) vs external (BrEaST+BUS-UCLM), by correctness:')
print(region_summary.to_string())
print('\nNote: internal/False is structurally empty -- Stratum 1 (Section 4) selected only')
print('consensus-correct BUSI-test images, so no internally-misclassified image exists in this sample.')

### 8.2 Padding Geometry

`get_padding_mask` reuses the exact scale/centring formula already verified in Section 2's `letterbox_mask`, not a new geometric assumption, the same one. Cross-verified against N02's own `compute_padding_fraction` formula, reproduced here for the check.

In [ ]:
def get_padding_mask(dataset, image_name, target_size=224):
    row = df_clean[(df_clean['dataset'] == dataset) & (df_clean['image_name'] == image_name)]
    assert len(row) == 1, f'Expected exactly one df_clean row for {dataset}/{image_name}, got {len(row)}.'
    h, w = int(row.iloc[0]['height']), int(row.iloc[0]['width'])

    scale = target_size / max(h, w)
    new_h, new_w = max(int(round(h * scale)), 1), max(int(round(w * scale)), 1)
    top = (target_size - new_h) // 2
    left = (target_size - new_w) // 2

    padding_mask = np.ones((target_size, target_size), dtype=np.uint8)
    padding_mask[top:top + new_h, left:left + new_w] = 0  # 0 = real content, 1 = padding
    return padding_mask


def formula_padding_fraction(h, w, target_size=224):
    """N02's own compute_padding_fraction, reproduced exactly for cross-verification."""
    scale = target_size / max(w, h)
    new_w, new_h = w * scale, h * scale
    real_area = new_w * new_h
    return 1 - (real_area / (target_size * target_size))


verify_sample = unique_images.sample(5, random_state=SEED)
for _, row in verify_sample.iterrows():
    dims = df_clean[(df_clean['dataset'] == row['dataset']) & (df_clean['image_name'] == row['image_name'])].iloc[0]
    h, w = int(dims['height']), int(dims['width'])

    pixel_frac = get_padding_mask(row['dataset'], row['image_name']).mean()
    formula_frac = formula_padding_fraction(h, w)

    assert abs(pixel_frac - formula_frac) < 0.01, (
        f"{row['dataset']}/{row['image_name']}: pixel-counted {pixel_frac:.4f} vs "
        f"N02 formula {formula_frac:.4f} -- disagreement exceeds rounding tolerance."
    )
    print(f"{row['dataset']}/{row['image_name']}: pixel={pixel_frac:.4f}, formula={formula_frac:.4f}  [OK]")

print('\nPadding geometry verified consistent with N02.')

### 8.3 Padding-Energy Density

For every one of the 361 heatmaps (no mask needed -- padding geometry is independent of lesion presence, so normal-class images are included here, unlike Section 7). `padding_density_ratio` near 1 = attention proportional to padding's share of the canvas (no special pull). Above 1 = disproportionate attention to non-content. Below 1 = padding actively avoided.

In [ ]:
padding_cache = {}
def get_cached_padding_mask(dataset, image_name):
    key = (dataset, image_name)
    if key not in padding_cache:
        padding_cache[key] = get_padding_mask(dataset, image_name)
    return padding_cache[key]

padding_records = []
for _, row in gradcam_manifest.iterrows():
    padding_mask = get_cached_padding_mask(row['dataset'], row['image_name'])
    heatmap = heatmaps[row['heatmap_id']]

    padding_pixel_fraction = float(padding_mask.mean())
    padding_energy_fraction = compute_ebpg(heatmap, padding_mask)

    if 0 < padding_pixel_fraction < 1:
        padding_density = padding_energy_fraction / padding_pixel_fraction
        content_density = (1 - padding_energy_fraction) / (1 - padding_pixel_fraction)
        density_ratio = padding_density / content_density if content_density > 0 else np.nan
    else:
        density_ratio = np.nan  # no padding present, or entirely padding -- undefined

    padding_records.append({
        'heatmap_id': row['heatmap_id'],
        'padding_pixel_fraction': padding_pixel_fraction,
        'padding_energy_fraction': padding_energy_fraction,
        'padding_density_ratio': density_ratio,
    })

padding_df = pd.DataFrame(padding_records)
attention_df = gradcam_manifest.merge(padding_df, on='heatmap_id')
attention_df['region'] = attention_df['dataset'].apply(lambda d: 'internal' if d == 'BUSI' else 'external')

n_undefined = attention_df['padding_density_ratio'].isna().sum()
print(f'Padding-energy computed for {len(attention_df)} heatmaps ({n_undefined} undefined -- no/all padding).')
print(f"\nActual physical padding amount by dataset (context for why raw fraction alone would mislead):")
print(attention_df.groupby('dataset')['padding_pixel_fraction'].mean().round(3).to_string())

### 8.4 Density Ratio: By Region and by Architecture

The architecture breakdown is the direct quantitative test of the Swin-Tiny edge-activation observation from Sections 3 and 6.

In [ ]:
density_valid = attention_df[attention_df['is_predicted'] & attention_df['padding_density_ratio'].notna()]

padding_by_region = density_valid.groupby(['region', 'correct'])['padding_density_ratio'].agg(
    ['count', 'mean', 'median', 'std']).round(3)
print('Padding-density ratio by region and correctness:')
print(padding_by_region.to_string())

padding_by_arch = density_valid.groupby('architecture')['padding_density_ratio'].agg(
    ['count', 'mean', 'median', 'std']).round(3)
print('\nPadding-density ratio by architecture:')
print(padding_by_arch.to_string())

swin_by_region = density_valid[density_valid['architecture'] == 'swin_tiny'].groupby('region')['padding_density_ratio'].agg(
    ['count', 'mean', 'median', 'std']).round(3)
print('\nSwin-Tiny only -- padding-density ratio by region:')
print(swin_by_region.to_string())

### 8.5 Cross-Reference with N04 & Save Summary

In [ ]:
ablation_summary = json.load(open(EVAL_DIR / 'ablation_clever_hans_summary.json'))
print('N04 ablation-based Clever Hans verdict, for reference:')
print(json.dumps(ablation_summary, indent=2))

gradcam_clever_hans_summary = {
    'padding_density_overall_mean': float(density_valid['padding_density_ratio'].mean()),
    'padding_density_by_region': padding_by_region.reset_index().to_dict(orient='records'),
    'padding_density_by_architecture': padding_by_arch.reset_index().to_dict(orient='records'),
    'swin_tiny_padding_density_by_region': swin_by_region.reset_index().to_dict(orient='records'),
    'note': ('Exploratory GradCAM++-based check for disproportionate attention on non-content '
             '(letterbox padding) regions, motivated by a qualitative observation during Section 3/6 '
             'review. Density ratio normalises for each dataset\'s differing physical padding amount '
             '(see 8.3). Complementary to, not a replacement for, the N04 ablation-arm Clever Hans '
             'investigation (inconclusive: 5 confirming / 4 disconfirming among externally-significant '
             'comparisons). Based on the Section 4 purposive sample (50 images, 361 heatmaps) -- '
             'descriptive, not inferential.'),
}

with open(GRADCAM_DIR / 'gradcam_clever_hans_summary.json', 'w') as f:
    json.dump(gradcam_clever_hans_summary, f, indent=2, default=str)
print(f'\nSaved: {GRADCAM_DIR / "gradcam_clever_hans_summary.json"}')

## 9. Radiologist Evaluation Materials

Selects ~20 images (5 per stratum) from the 50 already purposively-selected images, assigns a representative model per image via deterministic round-robin across all 6, and renders a **blinded** two-panel view (original + attention overlay) for each. Target class is shown (necessary context for a class-conditional heatmap); correctness, true label, dataset, model
identity, and stratum are withheld from the rendered image. A separate answer key, never shown to the radiologist, allows her responses to be joined back to full metadata later.

This produces the image materials only. The questionnaire/response-collection design is deferred.

In [ ]:
RADIOLOGIST_PACKET_DIR = GRADCAM_DIR / 'radiologist_packet'
RADIOLOGIST_PACKET_DIR.mkdir(parents=True, exist_ok=True)

answer_key_path = GRADCAM_DIR / 'radiologist_packet_answer_key.csv'

if answer_key_path.exists() and RADIOLOGIST_PACKET_LOCKED:
    print('Radiologist packet already exists and RADIOLOGIST_PACKET_LOCKED=True.')
    print('Her responses are already collected and keyed to these case_ids -- skipping regeneration')
    print('entirely to protect that mapping. Set RADIOLOGIST_PACKET_LOCKED = False in Section 1 to')
    print('override (only if you deliberately intend to discard her existing responses).')

    answer_key = pd.read_csv(answer_key_path)
    assert len(answer_key) == 20, f'Loaded answer key has {len(answer_key)} rows, expected 20.'

    saved_pngs = list(RADIOLOGIST_PACKET_DIR.glob('case_*.png'))
    assert len(saved_pngs) == 20, f'Answer key exists but only {len(saved_pngs)}/20 PNGs found -- investigate.'
    assert set(answer_key['case_id']) == {f'case_{i+1:02d}' for i in range(20)}, \
        'Loaded answer key case_ids do not match the expected case_01..case_20 set.'

    print('Verified: 20 rows, 20 PNGs on disk, case_ids consistent. Packet integrity confirmed intact.')
    SKIP_PACKET_GENERATION = True

else:
    if not answer_key_path.exists():
        print('No existing radiologist packet found -- generating for the first time.')
    else:
        print('WARNING: RADIOLOGIST_PACKET_LOCKED=False -- regenerating and OVERWRITING an existing')
        print('packet. If she already answered against the old case_ids, this WILL break that mapping.')
    SKIP_PACKET_GENERATION = False

### 9.1 Select Packet Images & Assign Representative Model

In [ ]:
if not SKIP_PACKET_GENERATION:
  PER_STRATUM_TARGET = 5

  packet_parts = []
  for stratum_id, label in STRATUM_LABELS.items():
      pool = stratum_membership[stratum_membership['stratum_id'] == stratum_id]
      n = min(PER_STRATUM_TARGET, len(pool))
      sampled = pool.sample(n=n, random_state=SEED)
      packet_parts.append(sampled)
      print(f'Packet / stratum {stratum_id} ({label}): pool={len(pool)}, selected={n}')

  packet_df = pd.concat(packet_parts, ignore_index=True).drop_duplicates(subset=['dataset', 'image_name']).reset_index(drop=True)
  print(f'\nTotal unique images in packet: {len(packet_df)}')

  # Deterministic round-robin model assignment across all 6 models
  packet_df = packet_df.sort_values(['dataset', 'class_name', 'image_name']).reset_index(drop=True)
  packet_df['assigned_model_id'] = [model_ids[i % len(model_ids)] for i in range(len(packet_df))]
  packet_df[['architecture', 'mode']] = packet_df['assigned_model_id'].str.rsplit('_', n=1, expand=True)

  print('\nModel assignment counts (round-robin coverage check):')
  print(packet_df['assigned_model_id'].value_counts().to_string())

### 9.2 Locate Heatmaps & Assign Blinded Case IDs

Always the predicted-class heatmap (`is_predicted == True`), per the Tier 1 design. Presentation order is shuffled, not stratum-ordered.

In [ ]:
if not SKIP_PACKET_GENERATION:
  def find_predicted_heatmap_id(dataset, image_name, architecture, mode):
      match = gradcam_manifest[
          (gradcam_manifest['dataset'] == dataset) & (gradcam_manifest['image_name'] == image_name) &
          (gradcam_manifest['architecture'] == architecture) & (gradcam_manifest['mode'] == mode) &
          (gradcam_manifest['is_predicted'])
      ]
      assert len(match) == 1, f'Expected 1 predicted-class heatmap for {dataset}/{image_name}/{architecture}/{mode}, got {len(match)}.'
      return match.iloc[0]['heatmap_id']

  packet_df['heatmap_id'] = packet_df.apply(
      lambda r: find_predicted_heatmap_id(r['dataset'], r['image_name'], r['architecture'], r['mode']), axis=1
  )

  packet_df = packet_df.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle presentation order
  packet_df['case_id'] = [f'case_{i+1:02d}' for i in range(len(packet_df))]

  print(f'Assigned {len(packet_df)} blinded case IDs, presentation order shuffled.')

### 9.3 Render & Save Blinded Cases

Two panels: original image, attention overlay. No mask boundary, no correctness indicator, no dataset/model/stratum label, only the target class the heatmap explains.

In [ ]:
if not SKIP_PACKET_GENERATION:
  def render_blinded_case(row):
      image_arr = load_final_image(row['dataset'], row['image_name'], row['class_name'])
      heatmap = heatmaps[row['heatmap_id']]
      input_np_float = np.float32(image_arr) / 255.0
      overlay = show_cam_on_image(input_np_float, heatmap, use_rgb=True)

      target_class_name = gradcam_manifest[gradcam_manifest['heatmap_id'] == row['heatmap_id']].iloc[0]['target_class_name']

      fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
      axes[0].imshow(image_arr); axes[0].set_title('Original', fontsize=10)
      axes[1].imshow(overlay); axes[1].set_title(f'Model attention\n(evidence considered for: {target_class_name})', fontsize=10)
      for ax in axes:
          ax.axis('off')
      fig.suptitle(row['case_id'], fontsize=11)
      plt.tight_layout()
      return fig


  for _, row in packet_df.iterrows():
      fig = render_blinded_case(row)
      fig.savefig(RADIOLOGIST_PACKET_DIR / f"{row['case_id']}.png", dpi=150, bbox_inches='tight')
      plt.close(fig)

  saved_files = list(RADIOLOGIST_PACKET_DIR.glob('case_*.png'))
  assert len(saved_files) == len(packet_df), f'{len(saved_files)} files on disk, expected {len(packet_df)}.'
  print(f'Saved and verified: {len(saved_files)} blinded case images in {RADIOLOGIST_PACKET_DIR}')

### 9.4 Answer Key (Not For Radiologist)

Full metadata per case, kept separate for joining her responses back later.

In [ ]:
if not SKIP_PACKET_GENERATION:
  answer_key = packet_df[['case_id', 'dataset', 'eval_data', 'image_name', 'class_name', 'architecture', 'mode', 'heatmap_id']].merge(
      gradcam_manifest[['heatmap_id', 'true_label', 'pred_label', 'correct', 'target_class_name']], on='heatmap_id'
  ).merge(
      stratum_membership[['dataset', 'image_name', 'stratum_id', 'stratum_label']].drop_duplicates(subset=['dataset', 'image_name']),
      on=['dataset', 'image_name'], how='left'
  )

  answer_key_path = GRADCAM_DIR / 'radiologist_packet_answer_key.csv'
  answer_key.to_csv(answer_key_path, index=False)

  print(f'Saved: {answer_key_path}')
  print(f'\nPacket composition ({len(answer_key)} cases):')
  print(answer_key[['case_id', 'stratum_label', 'correct']].to_string(index=False))

#### Clean Grad-CAM++ Panels for the Manuscript

In [ ]:
THESIS_FIG_DIR = RESULTS / 'thesis_figures'
THESIS_FIG_DIR.mkdir(parents=True, exist_ok=True)

def render_clean_panel(dataset, image_name, class_name, architecture, mode, save_path, dpi=300, size_inches=3.0):
    match = gradcam_manifest[
        (gradcam_manifest['dataset'] == dataset) & (gradcam_manifest['image_name'] == image_name) &
        (gradcam_manifest['architecture'] == architecture) & (gradcam_manifest['mode'] == mode) &
        (gradcam_manifest['is_predicted'])
    ]
    assert len(match) == 1, f'Expected 1 predicted-class heatmap for {dataset}/{image_name}/{architecture}/{mode}, got {len(match)}.'
    heatmap_id = int(match.iloc[0]['heatmap_id'])

    image_arr = load_final_image(dataset, image_name, class_name)
    heatmap = heatmaps[heatmap_id]
    input_np_float = np.float32(image_arr) / 255.0

    # image_weight=0.6 -> heatmap contributes (1-0.6)=0.4, matching the requested alpha=0.4
    overlay = show_cam_on_image(input_np_float, heatmap, use_rgb=True, image_weight=0.6)

    mask_result = get_lesion_mask(dataset, image_name, class_name)
    if mask_result['found']:
        mask_uint8 = (mask_result['mask'] * 255).astype(np.uint8)
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, (255, 255, 255), 2)

    fig = plt.figure(figsize=(size_inches, size_inches), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1])  # full-bleed, zero margin -- guarantees exact size_inches*dpi pixel output
    ax.imshow(overlay)
    ax.axis('off')
    fig.savefig(save_path, dpi=dpi)
    plt.close(fig)

    saved_img = Image.open(save_path)
    ebpg_val = localization_df[localization_df['heatmap_id'] == heatmap_id]['ebpg'].values
    ebpg_str = f'{ebpg_val[0]:.4f}' if len(ebpg_val) else 'N/A (no mask)'
    print(f'{save_path.name}: {saved_img.size[0]}x{saved_img.size[1]}px, heatmap_id={heatmap_id}, '
          f'EBPG={ebpg_str}, mask={"yes" if mask_result["found"] else "no"}')
    return heatmap_id


# --- Figure 1: TL vs Scratch, malignant (104).png, BUSI, all 6 models ---
fig1_targets = [
    ('resnet50', 'TL'), ('efficientnet_b0', 'TL'), ('swin_tiny', 'TL'),
    ('resnet50', 'scratch'), ('efficientnet_b0', 'scratch'), ('swin_tiny', 'scratch'),
]

print('=== Figure 1 panels (malignant (104).png, BUSI) ===')
for architecture, mode in fig1_targets:
    save_path = THESIS_FIG_DIR / f'fig1_{architecture}_{mode}.png'
    render_clean_panel('BUSI', 'malignant (104).png', 'malignant', architecture, mode, save_path)

# --- Figure 2: cross-dataset, ResNet50/TL only ---
print('\n=== Figure 2 panels (ResNet50/TL, cross-dataset) ===')
fig2_targets = [
    ('BUSI', 'malignant (104).png', 'fig2_busi.png'),
    ('BrEaST', 'case144.png', 'fig2_breast.png'),
    ('BUS-UCLM', 'COPE_012.png', 'fig2_bus_uclm.png'),
]

for dataset, image_name, fname in fig2_targets:
    class_name = gradcam_manifest[
        (gradcam_manifest['dataset'] == dataset) & (gradcam_manifest['image_name'] == image_name)
    ].iloc[0]['class_name']
    save_path = THESIS_FIG_DIR / fname
    render_clean_panel(dataset, image_name, class_name, 'resnet50', 'TL', save_path)


print('=== Verification ===')
all_files = sorted(THESIS_FIG_DIR.glob('fig*.png'))
assert len(all_files) == 9, f'Expected 9 files, found {len(all_files)}.'

for f in all_files:
    img = Image.open(f)
    status = 'PASS' if img.size[0] >= 800 and img.size[1] >= 800 else 'FAIL'
    print(f'[{status}] {f.name}: {img.size[0]}x{img.size[1]}px')

checks = [
    ('BUSI', 'malignant (104).png', 'resnet50', 'TL', 0.714),
    ('BrEaST', 'case144.png', 'resnet50', 'TL', 0.709),
    ('BUS-UCLM', 'COPE_012.png', 'resnet50', 'TL', 0.460),
]
for dataset, image_name, architecture, mode, claimed in checks:
    actual = localization_df[
        (localization_df['dataset'] == dataset) & (localization_df['image_name'] == image_name) &
        (localization_df['architecture'] == architecture) & (localization_df['mode'] == mode) &
        (localization_df['is_predicted'])
    ]['ebpg'].values[0]
    status = 'PASS' if abs(actual - claimed) < 0.001 else 'FAIL'
    print(f'[{status}] {dataset}/{image_name}: claimed={claimed}, actual={actual:.4f}')

print(f'\n9 clean panel images saved to {THESIS_FIG_DIR}')

## 10. Results Export & Completeness Check

Verifies every artifact this notebook produced actually exists on disk with the expected shape. Saves a file registry / README to `outputs/gradcam/` for reproducibility.

In [ ]:
EXPECTED_ROWS = {
    'stratum_membership.csv':          50,
    'selected_images.csv':             50,
    'gradcam_manifest.csv':            361,
    'gradcam_localization.csv':        313,
    'radiologist_packet_answer_key.csv': 20,
}

all_passed = True

for filename, expected in EXPECTED_ROWS.items():
    path = GRADCAM_DIR / filename
    if not path.exists():
        print(f'[FAIL] {filename}: MISSING')
        all_passed = False
        continue
    actual = len(pd.read_csv(path))
    status = 'PASS' if actual == expected else 'FAIL'
    print(f'[{status}] {filename}: {actual} rows (expected {expected})')
    if status == 'FAIL':
        all_passed = False

# npz array count
npz_check = np.load(GRADCAM_DIR / 'gradcam_heatmaps.npz')
n_arrays = len(npz_check.files)
status = 'PASS' if n_arrays == 361 else 'FAIL'
print(f'[{status}] gradcam_heatmaps.npz: {n_arrays} arrays (expected 361)')
all_passed &= (n_arrays == 361)

# File-count checks
for folder, expected_n in [('radiologist_packet', 20)]:
    n_files = len(list((GRADCAM_DIR / folder).glob('*.png')))
    status = 'PASS' if n_files == expected_n else 'FAIL'
    print(f'[{status}] {folder}/: {n_files} files (expected {expected_n})')
    all_passed &= (n_files == expected_n)

# Non-empty JSON checks
for filename in ['gradcam_clever_hans_summary.json']:
    path = GRADCAM_DIR / filename
    ok = path.exists() and path.stat().st_size > 0
    print(f'[{"PASS" if ok else "FAIL"}] {filename}: {"present, non-empty" if ok else "MISSING or empty"}')
    all_passed &= ok

if not all_passed:
    raise ValueError('Section 10 completeness check failed -- see FAILs above.')
print('\nAll N05 artifacts present and correctly sized.')

### 10.1 File Registry & README

In [ ]:
FILE_REGISTRY = [
    {'path': 'stratum_membership.csv', 'section': '4.6', 'description': 'One row per (image, stratum) pair, 50 rows, zero overlap across strata in this run.'},
    {'path': 'selected_images.csv', 'section': '4.6', 'description': 'Deduplicated unique images selected for GradCAM++ generation, 50 rows.'},
    {'path': 'gradcam_manifest.csv', 'section': '5.2', 'description': 'One row per generated heatmap (361), full metadata: model, target class, predicted/true flags, correctness.'},
    {'path': 'gradcam_heatmaps.npz', 'section': '5.3', 'description': 'Raw 224x224 float32 GradCAM++ arrays, keyed by heatmap_id, 361 arrays.'},
    {'path': 'gradcam_localization.csv', 'section': '7.2', 'description': 'EBPG and Otsu-IoU per heatmap where a lesion mask exists, 313 rows (48 skipped -- no mask).'},
    {'path': 'gradcam_clever_hans_summary.json', 'section': '8.5', 'description': 'Padding-energy density findings, complementary to N04 ablation-based Clever Hans verdict.'},
    {'path': 'radiologist_packet/', 'section': '9.3', 'description': '20 blinded case images (original + attention overlay), no correctness/dataset/model identifiers shown.'},
    {'path': 'radiologist_packet_answer_key.csv', 'section': '9.4', 'description': 'Full metadata per radiologist-packet case, kept separate for post-hoc joining.'},
]

readme_lines = ['# Notebook 05 -- GradCAM++ Results\n', 'Generated from actual file contents.\n']
for entry in FILE_REGISTRY:
    full_path = GRADCAM_DIR / entry['path']
    if entry['path'].endswith('/'):
        n = len(list((GRADCAM_DIR / entry['path'].rstrip('/')).glob('*.png')))
        detail = f'{n} files'
    elif full_path.suffix == '.csv':
        detail = f'{len(pd.read_csv(full_path))} rows'
    elif full_path.suffix == '.npz':
        detail = f'{len(np.load(full_path).files)} arrays'
    else:
        detail = 'present' if full_path.exists() else 'MISSING'
    readme_lines.append(f"- **{entry['path']}** ({entry['section']}) -- {detail}. {entry['description']}\n")

with open(GRADCAM_DIR / 'README.md', 'w') as f:
    f.writelines(readme_lines)

print(''.join(readme_lines))
print(f'\nSaved: {GRADCAM_DIR / "README.md"}')
print('\nNotebook 05 complete.')